# Evaluating Performance

In [33]:
# Importing packages
import pandas as pd
import torch
import os
os.environ["HF_HUB_DISABLE_PROGREvSS_BARS"] = "1"
from huggingface_hub import login
from transformers import AutoTokenizer, AutoModelForCausalLM, logging, AutoModel
logging.set_verbosity_error()
import numpy as np
from concurrent.futures import ThreadPoolExecutor
from dotenv import load_dotenv
import os
from torch import Tensor
import faiss 
import json
from beir.datasets.data_loader import GenericDataLoader
import torch.nn.functional as F
from beir.retrieval.evaluation import EvaluateRetrieval

In [6]:
### Loading Data ###
data_dir = "/work/mbouthil/datasets/msmarco"
corpus, dev_queries, dev_qrels = GenericDataLoader(data_folder=data_dir).load(split="dev")

100%|██████████| 8841823/8841823 [01:26<00:00, 102496.14it/s]


In [24]:
dev_info = [(key, value) for key, value in dev_queries.items()]

In [7]:
print(len(dev_queries))

6980


In [8]:
# Loading Index
index = faiss.read_index("/work/mbouthil/MMATH-CM-Research-Project/RAG/retrieval_data/passage_v1.index")

In [10]:
# Loading Query Encoder
device = "cuda" if torch.cuda.is_available() else "cpu"
tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")
query_encoder = AutoModel.from_pretrained(
    "/work/mbouthil/MMATH-CM-Research-Project/RAG/model_weights/query_encoder_v1"
).to(device)
query_encoder.eval()

def encode_query(query:str, batch_size:int=32) -> Tensor:

    queries = [query] if isinstance(query, str) else query
    embeddings = []

    with torch.no_grad():
        inputs = tokenizer(
            queries, 
            padding=True,
            truncation=True,
            return_tensors="pt",
            max_length=32
        ).to(device)

    emb = query_encoder(**inputs).last_hidden_state[:, 0]
    emb = F.normalize(emb, p=2, dim=-1)

    embeddings.append(emb.cpu())

    return torch.cat(embeddings, 0)

Loading weights: 100%|██████████| 199/199 [00:01<00:00, 183.84it/s, Materializing param=pooler.dense.weight]                               


In [23]:
dev_queries[0]

KeyError: 0

In [48]:
def batch_splits(item:list, batch_size:int=10):

    for i in range(0, len(item), batch_size):
        yield item[i:i + batch_size]

batches = batch_splits(dev_info)

In [ ]:
answers = {}

for batch in batches:

    q_ids, queries = zip(*batch)
    N = range(len(q_ids))
    
    q_emb = encode_query(queries).detach().cpu().numpy()
    scores, pids = index.search(q_emb, 10)
    
    pids = [[str(pid) for pid in pids[i]] for i in N]
    scores = [[float(score) for score in scores[i]] for i in N]
    
    batch_results = {
        q_ids[i]: dict(zip(pids[i], scores[i])) 
        for i in N
    }
    
    answers.update(batch_results)

In [ ]:
results = answers
print(results)

{'300674': {'4917600': 0.9988375306129456, '4259563': 0.9972956776618958, '2477943': 0.9972622394561768, '4381657': 0.996856153011322, '3289523': 0.9966931343078613, '6841855': 0.9966334104537964, '4882206': 0.9965806007385254, '5675016': 0.9965120553970337, '3588407': 0.9964436292648315, '2655322': 0.996015191078186}, '125705': {'138609': 0.9814088940620422, '4607525': 0.9776832461357117, '6737401': 0.9748872518539429, '7520247': 0.9736883640289307, '4255436': 0.9671425819396973, '6782459': 0.9624027013778687, '138605': 0.9603665471076965, '6737402': 0.956713855266571, '6782452': 0.9556042551994324, '2257895': 0.9531915187835693}, '94798': {'4877278': 0.9992272257804871, '8542791': 0.9991607666015625, '7999535': 0.9991222620010376, '2307878': 0.9990708231925964, '2447809': 0.9990589022636414, '8573601': 0.9990569353103638, '6848263': 0.9990399479866028, '6524885': 0.9990307688713074, '993410': 0.9990134239196777, '2896212': 0.9990048408508301}, '9083': {'2242670': 0.9972774982452393, 

In [51]:
ndcg, _map, recall, precision = EvaluateRetrieval.evaluate(
    dev_qrels,
    results, 
    k_values=[1, 3, 5, 10, 100, 1000]
)

mrr = EvaluateRetrieval.evaluate_custom(
    dev_qrels, 
    results, 
    k_values=[10], 
    metric="mrr"
)

print(f"MRR@10: {mrr['MRR@10']}") 

MRR@10: 0.0
